Lateral interactions on Cu(111) — on-the-fly site discovery

This notebook walks through the **on-the-fly KMC adsorption-site discovery**
machinery added to `autokmc.adsorbate` (see `dev/PLAN_adsorption_sites.md`).

We will:

1. Build a Cu(111) slab and run the **clean-surface bootstrap** that finds
   every unique adsorption site for atomic O.
2. Inspect what was registered on the graph: stable / unstable cliques,
   shared-atom neighbour map, and the per-site context cache.
3. Visualise the slab + sites with **plotly**, colouring sites by their
   live KMC state (`reactive`, `occupied`, `unstable`).
4. Trigger several KMC events with `register_adsorption` /
   `register_desorption` and watch the cache grow / `reactive` flags flip
   as new neighbour-occupancy contexts are discovered.

Heavy lifting (LBFGS relaxations) uses the deployed NequIP
Allegro model in this folder (`cpuhcocuau.nequip.pth`) — the same
calculator used in `dev_adsorption.ipynb`.

In [ ]:
"""
## 1. Imports and calculator

The new on-the-fly API lives next to the existing pipeline functions in
`autokmc.adsorbate`:

- `compute_neighbour_sites` — shared-surface-atom neighbour map
- `discover_context_site`   — relax (or cache-hit) one site under the
  current neighbour-occupancy context
- `register_adsorption` / `register_desorption` — KMC event hooks
- `update_reactive_flags`  — recompute the `reactive` flag

All cached state lives on the existing `nx.Graph` (`G.graph[...]`); no
external state object is needed.
"""

import os
import sys
import numpy as np
import networkx as nx

# Make `import autokmc` work when launching the notebook from this folder.
sys.path.insert(0, os.path.abspath('../../..'))

from ase.build import fcc111

from autokmc.surface import find_surface_atoms
from autokmc.graph   import build_graph
from autokmc.default_sites import (
    find_sites_for_element,
    reduce_sites_by_isomorphism,
    optimise_site_positions,
)
from autokmc.adsorbate import (
    optimise_unique_sites,
    register_adsorption,
    register_desorption,
    discover_context_site,
    ConnectivityStatus,
)


In [ ]:
# Load the deployed NequIP/Allegro calculator from this folder.
# (Same loader as dev_adsorption.ipynb.)
import torch
from nequip.ase import NequIPCalculator

_DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
_MODEL_FILE = 'asehcocuau.nequip.pt2' if _DEVICE == 'cuda' else 'cpuhcocuau.nequip.pth'
_MODEL_PATH = os.path.join(os.getcwd(), _MODEL_FILE)

calc = NequIPCalculator.from_compiled_model(
    compile_path=_MODEL_PATH, device=_DEVICE,
)
print(f'Calculator ready on {_DEVICE}: {_MODEL_FILE}')


In [ ]:
"""
## 2. Build the Cu(111) slab and run Stage 1 (geometric site enumeration)

`autokmc` separates pure-geometry work (Stage 1) from calculator work
(Stages 2/3) so we can iterate on site enumeration without paying the
ML cost. Stage 1 is:

1. `find_surface_atoms(slab, tag_atoms=True)` — writes `slab.arrays['surface']`
   (int8: 0=bulk, 1=surface, 2=adsorbate). Every downstream tool reads
   this column.
2. `build_graph(slab)` — atom-connectivity `nx.Graph` with cell + pbc on
   `G.graph` and (`element`, `position`, `type`, …) on every node.
3. `find_sites_for_element(G, 'O')` — every k-clique of the
   adsorbate-specific co-bonding graph. Stored under `G.graph['sites']['O']`.
4. `reduce_sites_by_isomorphism(..., n_shells=1)` — group cliques into
   iso-classes via 1-shell ego-graph isomorphism.
5. `optimise_site_positions(...)` — pure-scipy geometric placement of the
   adsorbate at the ideal bond-length distance from each clique atom.

We use a small `(3, 3, 4)` Cu(111) slab so each cell finishes in seconds.
"""

ELEMENT  = 'O'
N_SHELLS = 1   # iso-class shell depth (decision §10 of PLAN_adsorption_sites.md)

# 3x3 surface cell, 4 layers, 10 Å vacuum.
slab = fcc111('Cu', size=(3, 3, 4), vacuum=10.0, periodic=True)
find_surface_atoms(slab, tag_atoms=True)

G = build_graph(slab)
find_sites_for_element(G, ELEMENT, verbose=True)
reduce_sites_by_isomorphism(G, ELEMENT, n_shells=N_SHELLS, verbose=True)
optimise_site_positions(G, ELEMENT, verbose=True)

print('\nNodes in graph :', G.number_of_nodes())
print('Sites enumerated by k:', {k: len(v) for k, v in G.graph['sites'][ELEMENT].items()})


In [ ]:
"""
## 3. Stage 2/3 — calculator bootstrap on the clean surface

`optimise_unique_sites` does three things in one call:

1. Relaxes one **representative per iso-class** with the calculator.
2. Runs `check_connectivity` on the relaxed structure to decide whether
   the adsorbate stayed where we put it (`ConnectivityStatus.OK`) or
   migrated.
3. Registers an `AdsorptionSite` per **clique instance** (not just the
   representative) at `G.graph['adsorption_sites'][element][n_shells]`.
   With the on-the-fly extension each site now also carries:

   - `stable` — `True` iff the iso-class relaxed with `OK` connectivity.
   - `reactive` — initially equal to `stable` (clean surface).
   - `migrated_to` — destination clique for unstable sites (a hint for KMC).
   - `neighbour_sites` — cliques that share at least one surface atom.
   - `context_results` — per-site cache keyed by
     `frozenset(occupied_neighbour_cliques)`. Pre-seeded with the
     clean-surface (`frozenset()`) result.
   - `current_result` — the `SiteOptResult` whose `adsorption_energy`
     KMC reads to compute rates.
"""

# This single call performs all the per-iso-class LBFGS relaxations.
# fmax=0.10 / steps=200 keeps the demo fast; production should tighten.
optimise_unique_sites(
    G, ELEMENT, slab, calc,
    n_shells=N_SHELLS,
    fmax=0.10, steps=200,
    verbose=True,
)

sites = G.graph['adsorption_sites'][ELEMENT][N_SHELLS]
print(f'\n{len(sites)} AdsorptionSite objects registered.')


In [ ]:
"""
## 4. Inspect the bootstrap state

Quick summary of every site grouped by (k, stable, reactive). On a
clean Cu(111) the only stable hollow type for O is the fcc hollow, so
we expect every fcc clique to be stable+reactive and the others to be
either unstable (`migrated_to` set) or reactive at lower coordination.

We also print the neighbour-list size for one representative site to
sanity-check `compute_neighbour_sites` (a hollow on Cu(111) has
exactly 9 shared-atom neighbours: 3 hollows of the opposite type +
6 bridges along its 3 edges — actually 3 + 6 = 9, plus its 3 top
sites = 12 total).
"""

from collections import Counter

# Aggregate counts by (k, stable, reactive)
summary = Counter(
    (len(c), s.stable, s.reactive)
    for c, s in sites.items()
)
print('  k  stable  reactive  count')
print('  ---------------------------')
for (k, st, rx), n in sorted(summary.items()):
    print(f'  {k}  {str(st):>6}  {str(rx):>8}  {n:>5}')

# Neighbour count for one example hollow
any_hollow = next(c for c in sites if len(c) == 3)
print(f'\nExample hollow {set(any_hollow)} has '
      f'{len(sites[any_hollow].neighbour_sites)} shared-atom neighbours')


In [ ]:
"""
## 5. Plotly visualisation helper

We render a top-down view of the slab. Conventions:

- **Cu surface atoms** — small orange spheres at their (x, y, z).
- **Adsorption sites** — coloured markers at `site.ads_position`:
  - **green** = stable & reactive (KMC can fire adsorption here without
    triggering a new relaxation)
  - **yellow** = stable but **not** reactive (currently-occupied
    neighbour pattern has no cached result yet — `register_adsorption`
    will discover it on the fly)
  - **red** = currently `occupied`
  - **gray X** = unstable (the iso-class did not stay put on the clean
    surface; `migrated_to` is set)
- **Marker size** scales with `k` (top=small, bridge=medium, hollow=large)
  so the three site types are visually distinct.
- The figure title shows live counts so we can see them change after each
  KMC event.
"""

import plotly.graph_objects as go

_K_TO_LABEL = {1: 'top', 2: 'bridge', 3: 'hollow'}
_K_TO_SIZE  = {1: 6, 2: 9, 3: 13}

def _classify(site):
    """Map an AdsorptionSite to (state_label, color, symbol)."""
    if not site.stable:
        return 'unstable', 'lightgray', 'x'
    if site.occupied:
        return 'occupied', 'crimson', 'circle'
    if site.reactive:
        return 'reactive', 'mediumseagreen', 'circle'
    return 'idle (cached miss)', 'gold', 'circle-open'

def plot_state(G, element, n_shells, slab, title=''):
    """3-D plotly figure of the slab + sites in their current KMC state."""
    sites = G.graph['adsorption_sites'][element][n_shells]
    pos = slab.get_positions()
    syms = slab.get_chemical_symbols()

    # --- Cu surface atoms ------------------------------------------------
    surf_mask = slab.arrays['surface'] == 1
    cu_trace = go.Scatter3d(
        x=pos[surf_mask, 0], y=pos[surf_mask, 1], z=pos[surf_mask, 2],
        mode='markers', name='Cu (surface)',
        marker=dict(size=4, color='goldenrod', opacity=0.85),
        hovertext=[f'Cu #{i}' for i, m in enumerate(surf_mask) if m],
    )
    # Bulk Cu in fainter color for context
    bulk = ~surf_mask
    cu_bulk = go.Scatter3d(
        x=pos[bulk, 0], y=pos[bulk, 1], z=pos[bulk, 2],
        mode='markers', name='Cu (bulk)', showlegend=False,
        marker=dict(size=3, color='lightgray', opacity=0.35),
    )

    # --- Adsorption sites grouped by state -------------------------------
    # Group so each state appears as a single legend entry.
    groups: dict = {}
    for clique, site in sites.items():
        state, color, symbol = _classify(site)
        groups.setdefault((state, color, symbol), []).append((clique, site))

    site_traces = []
    counts = {}
    for (state, color, symbol), items in groups.items():
        xs, ys, zs, sz, txt = [], [], [], [], []
        for clique, s in items:
            p = s.ads_position
            xs.append(p[0]); ys.append(p[1]); zs.append(p[2])
            sz.append(_K_TO_SIZE[len(clique)])
            txt.append(
                f'{_K_TO_LABEL[len(clique)]} clique={set(clique)}<br>'
                f'state={state}<br>'
                f'E_ads={s.current_result.adsorption_energy:+.3f} eV'
            )
        counts[state] = len(items)
        site_traces.append(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='markers',
            name=f'{state} ({len(items)})',
            marker=dict(size=sz, color=color, symbol=symbol,
                        line=dict(color='black', width=0.5)),
            hovertext=txt, hoverinfo='text',
        ))

    # --- Title with live state counts -----------------------------------
    title_full = (
        title + '<br>'
        + '  '.join(f'{k}: {v}' for k, v in counts.items())
    )
    fig = go.Figure([cu_bulk, cu_trace, *site_traces])
    fig.update_layout(
        title=title_full,
        scene=dict(
            aspectmode='data',
            xaxis_title='x (Å)', yaxis_title='y (Å)', zaxis_title='z (Å)',
            camera=dict(eye=dict(x=0.0, y=-0.1, z=2.0)),  # near top-down
        ),
        legend=dict(itemsizing='constant'),
        margin=dict(l=0, r=0, t=70, b=0), height=600,
    )
    return fig


In [ ]:
"""
### 5a. Initial clean-surface plot

On the clean surface every **stable** site is **reactive** (its
`context_results[frozenset()]` was seeded by the bootstrap), so we
should see green dots everywhere except for any unstable iso-classes
(grey ✕).
"""

plot_state(G, ELEMENT, N_SHELLS, slab, title='Step 0 — clean surface').show()


In [ ]:
"""
## 6. KMC event #1 — adsorb at a hollow

We pick one stable hollow as our seed and call `register_adsorption`.
That function:

1. Marks the seed `occupied=True`, `reactive=False`.
2. Iterates over **every stable, vacant neighbour** of the seed and
   calls `discover_context_site` for it. Each such call:
    - looks the new occupancy key up in the per-site cache (miss),
    - falls back to the global iso-deduplicated cache
      (`G.graph['context_cache']`) — a miss the first time we ever see
      this neighbour topology, a **hit** thereafter for every
      symmetry-equivalent neighbour,
    - on a true miss runs LBFGS and stores the result in both caches.
3. Calls `update_reactive_flags` for the changed clique + its neighbours
   and returns the set of cliques whose `reactive` flag flipped.

After this cell every neighbour of the seed has a fresh
`current_result` reflecting the new neighbour-occupancy context, so
they go back to `reactive=True` (green) — but with a *different* energy
than the clean-surface bootstrap.
"""

# Pick the first stable hollow as the seed.
seed = next(c for c, s in sites.items() if s.stable and len(c) == 3)
print(f'Seed clique: {set(seed)}')
print(f'Bootstrap E_ads: {sites[seed].current_result.adsorption_energy:+.4f} eV')
print(f'Stable, vacant neighbours: '
      f'{sum(1 for c in sites[seed].neighbour_sites if sites[c].stable and not sites[c].occupied)}')

flipped = register_adsorption(
    G, ELEMENT, N_SHELLS, seed, slab, calc,
    fmax=0.10, steps=100,
)
print(f'\n{len(flipped)} cliques flipped their reactive flag during this event.')


In [ ]:
plot_state(G, ELEMENT, N_SHELLS, slab,
           title='Step 1 — one O adsorbed at a hollow').show()


In [ ]:
"""
Notice that the cache grew — we can read out exactly how many fresh
relaxations were needed and how many were saved by the iso-dedup
cache hits.
"""

def cache_stats(G, element, n_shells):
    """Return (n_global_entries, n_per_site_entries) for diagnostics."""
    ctx = G.graph.get('context_cache', {}).get(element, {}).get(n_shells, {})
    n_global = sum(len(v) for v in ctx.values())
    n_local  = sum(len(s.context_results) for s in
                   G.graph['adsorption_sites'][element][n_shells].values())
    return n_global, n_local

n_g, n_l = cache_stats(G, ELEMENT, N_SHELLS)
print(f'Global iso-dedup cache : {n_g} unique relaxations cached')
print(f'Per-site exact cache    : {n_l} (key, result) entries across all sites')
print(f'Sites: {len(sites)} → so on average each site has '
      f'{n_l / len(sites):.2f} cached contexts')


In [ ]:
"""
## 7. KMC event #2 — adsorb a second O at a non-neighbour hollow

Adsorbing far away from the first O means the new neighbour-occupancy
pattern around its own neighbours is the same one we discovered in
step 1 (one occupied hollow nearby). Because of the **iso-dedup
cache** every neighbour's discovery should now be a **cache hit** —
no new LBFGS runs, just lookups. We measure this by snapshotting the
cache size before and after.
"""

# Find a stable hollow that is NOT a neighbour of `seed` (clean local context).
second = next(
    c for c, s in sites.items()
    if s.stable and not s.occupied and len(c) == 3
    and c not in sites[seed].neighbour_sites
)
print(f'Second seed clique: {set(second)}')

n_g_before, _ = cache_stats(G, ELEMENT, N_SHELLS)
register_adsorption(
    G, ELEMENT, N_SHELLS, second, slab, calc,
    fmax=0.10, steps=100,
)
n_g_after, _ = cache_stats(G, ELEMENT, N_SHELLS)
print(f'Global cache grew by {n_g_after - n_g_before} entries during this event.')
print('(Lower = more iso-dedup hits.)')


In [ ]:
plot_state(G, ELEMENT, N_SHELLS, slab,
           title='Step 2 — two O atoms adsorbed').show()


In [ ]:
"""
## 8. KMC event #3 — adsorb a neighbour of the first seed

This event genuinely changes the local environment: the freshly
adsorbed atom now has the **first seed** as an occupied neighbour and
vice-versa. The discovery loop will fan out to *its* stable vacant
neighbours, some of which may now have **two** occupied neighbours —
a brand-new context that has to be relaxed for the first time.
"""

third = next(
    c for c in sites[seed].neighbour_sites
    if sites[c].stable and not sites[c].occupied and len(c) == 3
)
print(f'Third seed clique (neighbour of first seed): {set(third)}')

n_g_before, _ = cache_stats(G, ELEMENT, N_SHELLS)
register_adsorption(
    G, ELEMENT, N_SHELLS, third, slab, calc,
    fmax=0.10, steps=100,
)
n_g_after, _ = cache_stats(G, ELEMENT, N_SHELLS)
print(f'Global cache grew by {n_g_after - n_g_before} entries '
      f'(new contexts discovered = lateral interaction signal.)')

plot_state(G, ELEMENT, N_SHELLS, slab,
           title='Step 3 — third O adsorbed next to the first seed').show()


In [ ]:
"""
## 9. KMC event #4 — desorb the first seed

`register_desorption` is a **pure graph operation** (no calculator):
the cache stays valid because every entry is keyed by the occupancy
pattern, and we just re-evaluate `reactive` flags.

After desorption every neighbour falls back to whatever cached entry
matches its **new** occupancy pattern (e.g. zero occupied neighbours →
the original clean-surface bootstrap, or one occupied neighbour →
an entry from step 1 / step 3).
"""

n_g_before, n_l_before = cache_stats(G, ELEMENT, N_SHELLS)
flipped = register_desorption(G, ELEMENT, N_SHELLS, seed)
n_g_after, n_l_after = cache_stats(G, ELEMENT, N_SHELLS)

print(f'Global cache before -> after : {n_g_before} -> {n_g_after} '
      f'(no new relaxations expected on desorption)')
print(f'Per-site cache before -> after : {n_l_before} -> {n_l_after}')
print(f'Reactive flags flipped : {len(flipped)} cliques')

plot_state(G, ELEMENT, N_SHELLS, slab,
           title='Step 4 — first seed desorbed (cache reused)').show()


## 10. Summary

What we just exercised:

| Step | Event                                  | Calculator calls          | New cache entries |
|------|----------------------------------------|---------------------------|-------------------|
| 0    | bootstrap (Stage 2/3)                  | one per iso-class         | 1 per stable site (empty key) |
| 1    | adsorb seed                            | one per stable neighbour  | first instance of each new context |
| 2    | adsorb a non-neighbour                 | mostly **zero** (iso-dedup hits) | small |
| 3    | adsorb a neighbour of seed             | non-zero (genuinely new contexts) | larger |
| 4    | desorb seed                            | **zero**                  | zero (pure graph op) |

Production KMC will plug into this by reading `site.current_result.adsorption_energy`
for rate constants and calling `register_adsorption` / `register_desorption`
after every accepted move — the cache structure means the calculator
is only invoked when a genuinely new local environment is encountered.